# Governance gates: Three checks before running analyses

To deploy social science microdata in practice, what often blocks analyses is not modeling but three "can-we?" questions: Can people in this data be re-identified? Do we actually have the rights to scrape these sources and redistribute them? How do we account for generative AI used in the manuscript to journals? In bioinformatics these are typically post-hoc compliance checklists; in social science, they are gating decisions that determine whether research **can even start**—fail them, and even polished regressions may not be publishable and arguably should not be done.

These three questions each have established methods. **Identifiability** is measured with **k-anonymity**: combine several "quasi-identifiers" (like strata, sampling units, age bands), count the smallest group—`k` is the size of the smallest equivalence class. `k=1` means someone is unique in their combination and can be directly re-identified; larger `k` is safer. **Data licensing** is a per-source assessment of copyright and permissions: public domain, Creative Commons (CC), publisher text-mining (TDM), library/archive (GLAM), platform terms of service (ToS)—different sources grant vastly different scraping and redistribution rights, and absence of evidence of permission is not permission. **AI use disclosure** has one true red line, not "did you use AI," but **"adopted-but-unverified"**—AI-generated content inserted into a manuscript without anyone checking it.

This tutorial uses `socialverse` to thread these three gates into one governance chain: load microdata → declare analysis unit → run ethics gates (observe NO-GO, then remediate step-by-step to PASS) → license triage → AI disclosure audit → produce a governance dashboard → generate an audit trail. `socialverse` is an analysis library for social science that treats governance as a first-class citizen alongside DiD, complex sampling, and other analytical methods—each gate is a callable function with structured, traceable decisions. In current practice: k-anonymity usually relies on **ARX / sdcMicro** or hand-rolled `pandas`, licensing is judged by reading rightsstatements.org and journal ToS, AI disclosure is pasted from **ICMJE / COPE** templates. Here we collect them in one chain.

We use a built-in synthetic survey dataset: 300 respondents with strata `strata`, primary sampling units `psu`, six-item attitude scale `item1..item6`, sampling weight `weight`, exposure `exposure`, and outcome `outcome`. It is small enough and structured clearly so you can see the evidence for each gate.

In [1]:
import matplotlib
matplotlib.use("Agg")  # 无显示环境:图直接写文件,必须在 import pyplot 之前设定

import json
import os

import pandas as pd

import socialverse as sv
from socialverse import datasets as ds

# 图保存到 notebook 同目录,这样无论从哪个 cwd 运行都对得上 ![](fig.png) 相对引用
try:
    _HERE = os.path.dirname(os.path.abspath(__file__))
except NameError:  # 交互式 Jupyter 里没有 __file__
    _HERE = os.getcwd()

def _fig(name):
    return os.path.join(_HERE, name)

print("socialverse version:", sv.__version__)

socialverse version: 0.1.0


## Loading the data

First, read in the survey data and take a look. The quasi-identifier candidates we care about are: `strata` (3 strata) and `psu` (19 primary sampling units)—their combination determines how easy it is to single someone out from the population. The attitude scale and weight columns are not used in this tutorial, but their presence illustrates how readily-available person-locating fields are everywhere in real microdata.

In [2]:
df = ds.load_survey()
print("调查数据 shape:", df.shape)
print("strata 取值:", sorted(df["strata"].unique().tolist()),
      " · psu 取值数:", df["psu"].nunique())
df.head()

调查数据 shape: (300, 11)
strata 取值: [1, 2, 3]  · psu 取值数: 19


,item1,item2,item3,item4,item5,item6,weight,strata,psu,exposure,outcome
0,4,2,3,2,4,3,2.002,2,16,1,0
1,3,2,4,3,3,2,2.260,1,11,0,1
2,4,4,4,4,3,5,2.974,2,13,1,1
3,1,4,2,3,3,2,1.894,2,14,1,1
4,3,3,3,2,2,2,1.602,1,6,1,1


## Declaring the analysis unit

The first step in the ethics gate is not computing k, but asking clearly: "What is the analysis unit?"—individuals? households? countries? This determines whether the research **involves human subjects**. We construct a `StudyState` (analogous to AnnData: a container holding study state throughout, organized by slots), register the data in `sources`, and declare the analysis unit as `row` (one row = one respondent = human subject). These two registrations are entry conditions for the subsequent `data_use_check` and `ethics_check`.

In [3]:
st = sv.StudyState()
st.write("sources", "datasets", df)   # 供数据合规闸门做许可分诊
st.write("design", "unit", "row")     # 分析单位:一行 = 一位受访者 = 人类受试者
print("已填槽:", st.populated())

已填槽: {'sources': ['datasets'], 'design': ['unit']}


## Ethics gate: Bare state reports NO-GO as-is

`ethics_check` runs four checks and collapses them to one decision: `PASS / FIX / NO-GO`. **IRB** (is human subjects review documented?), **informed consent** (is there a consent basis?), **identifiability** (true k-anonymity calculation on declared quasi-identifiers, default threshold `k_threshold=5`), **data minimization** (are direct identifiers removed?). The aggregation rule is straightforward: any NO-GO makes the whole decision NO-GO.

In our first version, we **declare nothing**, only pass `strata+psu` as quasi-identifiers for k calculation, and see what decision the gate renders for a bare state with no governance work.

In [4]:
sv.gov.ethics_check(st, data=df, quasi_identifiers=["strata", "psu"])  # k_threshold 默认 5

ethics_v1 = st.governance["ethics"]
print("伦理判决:", ethics_v1["verdict"])
print("\n四项检查:")
for c in ethics_v1["checks"]:
    print(f"  [{c['status']:<5}] {c['check']:<12} — {c['detail']}")

伦理判决: NO-GO

四项检查:
  [NO-GO] irb          — human subjects but no IRB determination recorded
  [NO-GO] consent      — human subjects but no consent basis recorded
  [FIX  ] k_anonymity  — k=2 < threshold 5: generalize/suppress QIs or coarsen to reach k≥5
  [FIX  ] minimization — confirm direct identifiers dropped and variables minimized to need


The decision is **NO-GO**. IRB and consent are both NO-GO (human subjects present but no documented review/consent), data minimization is FIX, and k-anonymity is FIX—because the smallest equivalence class under `strata+psu` is only **k=2** individuals, below the threshold of 5. This is the value of governance as a first-class citizen: before you write your first analysis code, the gate structurally lays out "why we cannot proceed now," rather than waiting for a reviewer or IRB to reject you later. Below, see what that k-anonymity calculation actually produced.

In [5]:
print("k-匿名细节(真实计算 df.groupby(QI).size().min(),非占位):")
print(json.dumps(ethics_v1["k_anonymity"], ensure_ascii=False, indent=2))

k-匿名细节(真实计算 df.groupby(QI).size().min(),非占位):
{
  "quasi_identifiers": [
    "strata",
    "psu"
  ],
  "n_records": 300,
  "n_equivalence_classes": 57,
  "n_unique_records": 0,
  "share_unique": 0.0,
  "k_threshold": 5,
  "k": 2
}


The 300 records are partitioned into 57 equivalence classes by `strata × psu`, with the smallest class containing only 2 individuals (`k=2`) and no unique singleton records (`n_unique_records=0`). So the problem is not "someone is directly exposed," but "the granularity is too fine—just short of the threshold"—a problem remedied by **generalizing quasi-identifiers**.

## Remedying the ethics gate: generalize quasi-identifiers + document IRB/consent → PASS

NO-GO is not the end; it is a to-do list. We remediate each item: record IRB determination as `exempt` (exempted review), record consent basis as `informed`, confirm direct identifiers are removed with `minimized=True`; for k-anonymity, **generalize** quasi-identifiers from `strata+psu` to just `strata`—the standard k-anonymity remedy (generalizing/suppressing quasi-identifiers) to trade finer granularity for a larger minimum equivalence class. All four are real governance decisions by the researcher, declared to the gate via keyword parameters; after recalculation, it should render PASS.

In [6]:
st_fixed = sv.StudyState()
st_fixed.write("sources", "datasets", df)
st_fixed.write("design", "unit", "row")

sv.gov.ethics_check(
    st_fixed,
    data=df,
    quasi_identifiers=["strata"],   # 粗化:丢掉 psu(k-匿名的泛化补救)
    irb="exempt",                   # 记录 IRB 裁定
    consent="informed",             # 记录同意基础
    minimized=True,                 # 已删除直接标识符
    k_threshold=5,
)

ethics_v2 = st_fixed.governance["ethics"]
print("补救后判决:", ethics_v2["verdict"])
print("\n四项检查:")
for c in ethics_v2["checks"]:
    print(f"  [{c['status']:<5}] {c['check']:<12} — {c['detail']}")

补救后判决: PASS

四项检查:
  [PASS ] irb          — IRB determination on record: exempt
  [PASS ] consent      — consent basis: informed
  [PASS ] k_anonymity  — k=97 ≥ threshold 5
  [PASS ] minimization — direct identifiers removed / analysis restricted to needed variables


All four items green, overall **PASS**. The effect of generalization is intuitive: when quasi-identifiers shrink from `strata+psu` to just `strata`, the minimum equivalence class jumps from 2 to 97 individuals—well above the threshold of 5.

In [7]:
print(f"k-匿名:粗化前 (strata+psu) k={ethics_v1['k_anonymity']['k']}"
      f"  →  粗化后 (strata) k={ethics_v2['k_anonymity']['k']}(≥ 阈值 5,PASS)")

k-匿名:粗化前 (strata+psu) k=2  →  粗化后 (strata) k=97(≥ 阈值 5,PASS)


### Red line: k=1 direct identifiability is hard NO-GO

Not all k problems can be fixed by generalization. The extreme of k-anonymity is **k=1**: quasi-identifier combinations that belong to only one person, who can be directly re-identified. We artificially create a unique row identifier `resp_id` as a quasi-identifier—even with IRB, consent, and minimization all declared, the gate renders **NO-GO**, not a remediable FIX, because "300 singleton records" is structural disclosure that generalization cannot easily remedy. This shows the gate's stratification of identifiability: `k ≥ threshold` = PASS, `1 < k < threshold` = FIX, `k ≤ 1` = NO-GO.

In [8]:
df_leak = df.copy()
df_leak["resp_id"] = range(len(df_leak))   # 唯一直接标识符,人为制造 k=1

st_leak = sv.StudyState()
st_leak.write("sources", "datasets", df_leak)
st_leak.write("design", "unit", "row")
sv.gov.ethics_check(st_leak, data=df_leak, quasi_identifiers=["resp_id"],
                    irb="exempt", consent="informed", minimized=True)

leak = st_leak.governance["ethics"]
k_check = next(c for c in leak["checks"] if c["check"] == "k_anonymity")
print("判决:", leak["verdict"], "· k =", leak["k_anonymity"]["k"],
      "· 单例记录数:", leak["k_anonymity"]["n_unique_records"])
print("k-匿名检查:", k_check["status"], "—", k_check["detail"])

判决: NO-GO · k = 1 · 单例记录数: 300
k-匿名检查: NO-GO — k=1: unique records are directly re-identifiable (300 singletons)


## Data licensing: five-bucket triage

In social science, "Can I scrape this data and redistribute it?" is assessed per source. `data_use_check` triages each source into **five buckets** (most to least restrictive): `platform_tos` (platform terms of service), `publisher_tdm` (publisher text-mining license), `glam` (library/archive), `cc` (Creative Commons), `public_domain` (public domain). From the license string it derives `can_scrape` (can we scrape?), `redistribution` (can we redistribute?), `attribution` (is attribution required?), and NC/ND/SA obligation markers. Start with the simplest case: a clean `CC-BY-4.0` license.

In [9]:
sv.gov.data_use_check(st_fixed, license="CC-BY-4.0")

du = st_fixed.governance["data_use"]
print("单源 (CC-BY-4.0) 分诊:")
print("  桶:", du["bucket"], "· 可抓取:", du["can_scrape"],
      "· 再分发:", du["redistribution"], "· 需署名:", du["attribution"])
print("  说明:", du["per_source"][0]["note"])

单源 (CC-BY-4.0) 分诊:
  桶: cc · 可抓取: True · 再分发: share_alike_or_by · 需署名: True
  说明: redistribution allowed under the specific CC terms — honor BY / SA / NC / ND obligations of the exact version


CC-BY falls into the `cc` bucket: scrapable, redistributable under CC terms (`share_alike_or_by`), attribution required. This is what "good data" should look like. Next, two more common and thornier cases.

### Unknown license falls to most restrictive bucket

Governance's default posture is safe-by-default: **absence of evidence is not permission**. Pass an empty license string and watch the gate default it to the strictest `platform_tos` bucket—`can_scrape=False`, redistribution `prohibited`, with a clear to-do: clarify rights before scraping or sharing.

In [10]:
st_unknown = sv.StudyState()
st_unknown.write("sources", "datasets", df)
sv.gov.data_use_check(st_unknown, license="")   # 许可未知

du_u = st_unknown.governance["data_use"]
print("桶:", du_u["bucket"], "· 可抓取:", du_u["can_scrape"],
      "· 再分发:", du_u["redistribution"])
print("flags:")
for f in du_u["flags"]:
    print("  -", f)

桶: platform_tos · 可抓取: False · 再分发: prohibited
flags:
  - UNKNOWN licence — defaulted to strictest (platform_tos); resolve rights before scraping or sharing


### Multiple sources: "weakest link" aggregation

Real projects mix multiple sources. The gate triages each source, then takes the **intersection** of rights: if even one source cannot be scraped, the whole dataset cannot; redistribution rights are the most restrictive tier; NC/ND/SA obligation markers take the union. Below we mix four typical sources—public-domain census, Twitter platform ToS, publisher TDM, CC-BY-NC-SA images—and see how the weakest link pulls the whole dataset down.

In [11]:
st_multi = sv.StudyState()
st_multi.write("sources", "datasets", df)
sv.gov.data_use_check(st_multi, license={
    "census_pums": "Public Domain (US government work)",
    "twitter_api": "Twitter Developer Policy / platform ToS",
    "journal_tdm": "Publisher text-and-data-mining licence",
    "cc_images":   "CC-BY-NC-SA 4.0",
})

du_m = st_multi.governance["data_use"]
pd.DataFrame([
    {"来源": t["source"], "桶": t["bucket"],
     "可抓取": t["can_scrape"], "再分发": t["redistribution"]}
    for t in du_m["per_source"]
])

,来源,桶,可抓取,再分发
0,census_pums,public_domain,True,unrestricted
1,twitter_api,platform_tos,False,prohibited
2,journal_tdm,publisher_tdm,True,derived_only
3,cc_images,cc,True,share_alike_or_by


Source by source: census is public domain (most permissive), CC images can be used under CC terms, but Twitter falls into platform ToS (cannot scrape). Taking the intersection, the weakest link pulls the dataset to the strictest tier:

In [12]:
print("整盘(最弱环求交):")
print("  涉及的桶:", du_m["bucket"])
print("  can_scrape(全部可抓才为真):", du_m["can_scrape"])
print("  redistribution(取最严):", du_m["redistribution"])
print("  义务标记(并集上浮):")
for f in du_m["flags"]:
    print("   -", f)

整盘(最弱环求交):
  涉及的桶: ['cc', 'platform_tos', 'public_domain', 'publisher_tdm']
  can_scrape(全部可抓才为真): False
  redistribution(取最严): prohibited
  义务标记(并集上浮):
   - NoDerivatives clause — cannot redistribute modified versions
   - NonCommercial clause — no commercial reuse
   - ShareAlike clause — derivatives inherit the same licence


If even one source is Twitter, the whole dataset is `can_scrape=False`, redistribution `prohibited`—regardless of how permissive the other three sources are. This is the "weakest link": compliance looks to the floor of rights, not the ceiling.

## AI use disclosure: audit the "adopted-but-unverified" red line

Journals now universally require disclosure of generative AI use (ICMJE, COPE, Nature each have policies), but the real research integrity red line is not "did you use AI," but **"adopted-but-unverified"**—AI-generated content inserted directly into a manuscript without anyone checking it. `ai_use_disclosure` audits a stage-by-stage AI use log, specifically catching this red line, and renders a disclosure statement keyed to your target journal policy family that you can paste directly into your manuscript. First version: a log **with the red line**—AI output in the analysis stage was adopted but unverified, using the ICMJE policy family.

In [13]:
sv.gov.ai_use_disclosure(
    st_fixed,
    ai_log=[
        {"stage": "analysis", "tool": "LLM", "accepted": True,  "verified": False},  # 红线!
        {"stage": "drafting", "tool": "LLM", "accepted": True,  "verified": True},
    ],
    policy="ICMJE",
)

disc = st_fixed.governance["ai_disclosure"]
print("审计状态:", disc["audit"]["status"], "·", disc["audit"]["detail"])
print("政策族:", disc["policy"], "→", disc["policy_family"])
print("\n『已采纳但未核验』命中:")
for u in disc["audit"]["unverified"]:
    print("  -", u["stage"], "/", u["tool"])

审计状态: ESCALATE · 1 AI contribution(s) accepted without verification — verify or remove before submission (research-integrity red line)
政策族: ICMJE → icmje

『已采纳但未核验』命中:
  - analysis / LLM


The audit returns `ESCALATE`: the analysis entry is flagged and must be verified or removed before submission. The gate also organizes the full log into a table (which can go directly in the manuscript appendix), with the red-line row marked `accepted-but-unverified`:

In [14]:
st_fixed.artifacts["tables"]

,stage,tool,accepted,verified,flag
0,analysis,LLM,True,False,accepted-but-unverified
1,drafting,LLM,True,True,


Regardless of whether the audit flags the red line, the gate renders a disclosure statement you can paste into your manuscript, with wording keyed to the policy family:

In [15]:
print("ICMJE 政策族的披露声明:\n")
print(disc["statement"])

ICMJE 政策族的披露声明:

During the preparation of this work the author(s) used LLM in order to assist with analysis, drafting. After using LLM, the author(s) reviewed and edited the content as needed and take(s) full responsibility for the content of the publication. Generative AI was not listed as an author.


### Switch policy and verify all: down to PASS

Mark the same work as **all verified**, switch the policy family to **COPE**—the audit downgrades from `ESCALATE` to `PASS`, and the statement automatically switches to COPE wording. This illustrates the gate's two dimensions: red-line audit handles **integrity**, policy-family rendering handles **format**, and they do not interfere.

In [16]:
st_clean = sv.StudyState()
sv.gov.ai_use_disclosure(
    st_clean,
    ai_log=[
        {"stage": "copyediting", "tool": "Claude", "accepted": True, "verified": True},
        {"stage": "code review", "tool": "Claude", "accepted": True, "verified": True},
    ],
    policy="COPE",
)
disc_c = st_clean.governance["ai_disclosure"]
print("审计状态:", disc_c["audit"]["status"], "·", disc_c["audit"]["detail"])
print("政策族:", disc_c["policy"], "→", disc_c["policy_family"])
print("\nCOPE 政策族的披露声明:\n")
print(disc_c["statement"])

审计状态: PASS · all accepted AI contributions were human-verified
政策族: COPE → cope

COPE 政策族的披露声明:

The author(s) declare(s) the use of generative AI and AI-assisted technologies in the writing process. Specifically, Claude were used to assist with code review, copyediting. All AI-assisted output was verified by the author(s), who take(s) full responsibility for the integrity of the work. AI tools do not meet authorship criteria and are not credited as authors.


## Governance dashboard: two gates at a glance

Render the key evidence from both numeric gates into a dashboard so reviewers and collaborators see compliance posture instantly: left shows how k-anonymity improves as quasi-identifiers are progressively generalized, crossing the threshold to green; right shows each multi-source license's scrape/redistribution rights. The three k values on the left are **actually recomputed** (not hard-coded)—we run `ethics_check` again for each generalization level. Save as PNG in the same directory and embed with Markdown.

In [17]:
import matplotlib.pyplot as plt

# 复用包内的 CJK 字体探测(与 sv.pl 一致):有则用,无则回退 DejaVu(不报错)
try:
    from socialverse.pl._figure import _cjk_fonts
    plt.rcParams["font.sans-serif"] = _cjk_fonts() + ["DejaVu Sans"]
    plt.rcParams["axes.unicode_minus"] = False
except Exception:  # pragma: no cover
    pass

# 左图数据:k 随准标识符粗化的阶梯(每档都真实重算一次 ethics_check)
qi_ladder = [("strata+psu", ["strata", "psu"]), ("psu", ["psu"]), ("strata", ["strata"])]
ks = []
for _, qi in qi_ladder:
    _s = sv.StudyState(); _s.write("sources", "datasets", df); _s.write("design", "unit", "row")
    sv.gov.ethics_check(_s, data=df, quasi_identifiers=qi,
                        irb="exempt", consent="informed", minimized=True)
    ks.append(_s.governance["ethics"]["k_anonymity"]["k"])

# 右图数据:多源许可桶的再分发权利等级(0=禁止 … 4=无限制)
redist_rank = {"prohibited": 0, "derived_only": 1, "conditional": 2,
               "share_alike_or_by": 3, "unrestricted": 4}
sources = du_m["per_source"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.2))

# --- 左:k-匿名粗化阶梯 ---
labels = [lbl for lbl, _ in qi_ladder]
colors = ["#c0392b" if k < 5 else "#27ae60" for k in ks]
bars = ax1.bar(labels, ks, color=colors, edgecolor="#222", linewidth=0.6)
ax1.axhline(5, ls="--", color="#333", lw=1.2)
ax1.text(len(labels) - 0.5, 5.6, "k=5 阈值", ha="right", fontsize=9, color="#333")
for b, k in zip(bars, ks):
    ax1.text(b.get_x() + b.get_width() / 2, k + 1.5, str(k), ha="center", fontsize=11, fontweight="bold")
ax1.set_ylabel("k-匿名(最小等价类规模)")
ax1.set_xlabel("准标识符集合(自左向右逐步粗化)")
ax1.set_title("① 伦理闸门:k-匿名随粗化而改善")
ax1.set_ylim(0, max(ks) * 1.25)
ax1.grid(axis="y", alpha=0.25)

# --- 右:许可桶再分发权利 ---
names = [t["source"] for t in sources]
ranks = [redist_rank.get(t["redistribution"], 0) for t in sources]
scrape_ok = [t["can_scrape"] for t in sources]
bcolors = ["#27ae60" if ok else "#c0392b" for ok in scrape_ok]
y = range(len(names))
ax2.barh(list(y), ranks, color=bcolors, edgecolor="#222", linewidth=0.6)
ax2.set_yticks(list(y))
ax2.set_yticklabels(names)
ax2.set_xticks(list(redist_rank.values()))
ax2.set_xticklabels(["禁止", "仅衍生", "有条件", "署名/相同", "无限制"], rotation=20, fontsize=8)
ax2.set_xlabel("再分发权利(越右越宽松)")
ax2.set_title("② 数据合规:多源许可桶(绿=可抓取)")
for i, t in enumerate(sources):
    ax2.text(ranks[i] + 0.06, i, t["bucket"], va="center", fontsize=8, color="#333")
ax2.set_xlim(0, 4.8)
ax2.grid(axis="x", alpha=0.25)

fig.suptitle("研究治理仪表盘:伦理闸门 + 数据合规", fontsize=13, fontweight="bold")
fig.tight_layout(rect=(0, 0, 1, 0.96))
fig.savefig(_fig("fig_governance.png"), dpi=150, bbox_inches="tight")  # PNG → bbox tight
plt.close(fig)

# 把图元信息也记进 StudyState 的 artifacts 槽(与 sv.pl 的产物形态一致)
st_fixed.write("artifacts", "figures", {
    "governance_dashboard": {"path": _fig("fig_governance.png"), "dpi": 150,
                             "note": "k-匿名粗化阶梯 + 多源许可桶再分发权利"}
})
print("已保存图:", st_fixed.artifacts["figures"]["governance_dashboard"]["path"])
print("k 阶梯 (strata+psu → psu → strata):", ks)

已保存图: /Users/fernandozeng/Desktop/analysis/omicos-project/socialverse-worktrees/tutorials-rewrite/notebooks/fig_governance.png
k 阶梯 (strata+psu → psu → strata): [2, 10, 97]


![Research governance dashboard](fig_governance.png)

The left graph compresses the entire remediation logic into three bars: coarser quasi-identifiers yield larger minimum equivalence classes. `strata+psu` (k=2, red) → `psu` (k=10, red) → `strata` (k=97, green) crosses the threshold to green. On the right, only Twitter is red, yet it drags the whole dataset to "not scrapable"—the weakest link.

## Auditable evidence chain

This section takes a light look at the key difference between `socialverse` and typical compliance scripts. As the governance chain runs, `StudyState` automatically builds a ledger: which function each gate used, which slots it read, where it wrote its decision. `summary()` shows filled slots and steps; the `governance` slot is a one-stop compliance receipt—all three conclusions ("can we release? / can we redistribute? / how do we disclose?") live here, each traceable through the ledger back to the raw data or log it read. In social science, "which step and which data the conclusion came from" often matters as much as the conclusion itself.

In [18]:
print(st_fixed.summary())

print("\ngovernance 槽的三道闸门判决(一站式合规回执):")
print("  ethics.verdict       :", st_fixed.governance["ethics"]["verdict"])
print("  data_use.bucket      :", st_fixed.governance["data_use"]["bucket"],
      "· can_scrape:", st_fixed.governance["data_use"]["can_scrape"])
print("  ai_disclosure.status :", st_fixed.governance["ai_disclosure"]["audit"]["status"])

StudyState
  sources: datasets
  design: unit
  evidence: provenance
  governance: ethics, data_use, ai_disclosure
  artifacts: tables, figures
  provenance: 3 step(s)

governance 槽的三道闸门判决(一站式合规回执):
  ethics.verdict       : PASS
  data_use.bucket      : cc · can_scrape: True
  ai_disclosure.status : ESCALATE


The ledger records each step's contract line by line—what it read (`requires`), what it produced (`produces`)—so any compliance conclusion can be traced back all the way. This trajectory can go directly into the manuscript's methods/ethics section and IRB materials.

In [19]:
for r in st_fixed.provenance:
    req = ", ".join(f"{s}[{','.join(k)}]" for s, k in r["requires"].items()) or "∅"
    pro = ", ".join(f"{s}[{','.join(k)}]" for s, k in r["produces"].items()) or "∅"
    print(f"step {r['step']}: {sv.utils._friendly(r['function'])}")
    print(f"        requires {req}")
    print(f"        produces {pro}")

step 1: sv.gov.ethics_check
        requires design[unit]
        produces governance[ethics]
step 2: sv.gov.data_use_check
        requires sources[datasets]
        produces governance[data_use]
step 3: sv.gov.ai_use_disclosure
        requires ∅
        produces governance[ai_disclosure], artifacts[tables], evidence[provenance]


## Summary

We have walked a complete governance chain: declare unit → ethics gate (NO-GO → remediate → PASS, including k=1 red line) → license five-bucket triage (per-source / unknown defaults to strictest / multi-source weakest link) → AI disclosure audit and statement → dashboard → evidence chain. This mirrors real compliance practice scattered across tools: k-anonymity via **ARX / sdcMicro** or hand-rolled `pandas`, licensing via reading rightsstatements.org and journal TDM terms, AI disclosure pasted from **ICMJE / COPE** templates.

Compared to these scattered approaches, `socialverse` adds two things: governance is a **first-class citizen gate that actually stops you** (k-anonymity returns PASS/FIX/NO-GO, unknown licenses default to strictest bucket, AI disclosure specifically flags "adopted-but-unverified"), not an optional checklist; and the decisions from all three gates automatically flow into one auditable evidence chain, linked to your downstream analytical conclusions. The next tutorial, [09_literature_citation](09_literature_citation.ipynb), turns to literature and citation: search, triple-database verification to catch hallucinated citations, manuscript citation audit.